# 01 - Text Preprocessing, Normalization & Tokenization

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain **why preprocessing matters** in NLP and how noisy text degrades model performance.
- Apply a range of **text normalization** techniques (lowercasing, URL/email removal, HTML stripping, emoji handling, number handling, contraction expansion, Unicode normalization).
- Handle **punctuation** in a context-dependent way.
- Implement three levels of **tokenization**: whitespace, word-level (rule-based), and sentence-level.
- Build a **reusable preprocessing pipeline** and apply it to real-world messy text.

## Prerequisites

- Basic Python (strings, regex, lists).
- Familiarity with regular expressions (`re` module).

## Table of Contents

1. [Why Preprocessing Matters](#1-why-preprocessing-matters)
2. [Text Normalization Steps](#2-text-normalization-steps)
   - 2.1 Lowercasing
   - 2.2 Removing URLs, Emails, HTML Tags
   - 2.3 Handling Emojis
   - 2.4 Number Handling
   - 2.5 Expanding Contractions
   - 2.6 Unicode Normalization
3. [Punctuation Handling](#3-punctuation-handling)
4. [Tokenization](#4-tokenization)
   - 4.1 Whitespace Tokenization
   - 4.2 Word Tokenization (Rule-Based)
   - 4.3 Sentence Tokenization
5. [Using src/utils/text_preprocessing.py](#5-using-srcutilstext_preprocessingpy)
6. [Building a Preprocessing Pipeline](#6-building-a-preprocessing-pipeline)
7. [Common Mistakes & Debugging Tips](#7-common-mistakes--debugging-tips)
8. [Exercises](#8-exercises)
9. [Exercise Solutions](#9-exercise-solutions)

In [1]:
# ---- Environment setup ----
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import re
import string
import unicodedata
from collections import Counter
from typing import List, Callable

print("Setup complete.")

Setup complete.


## 1. Why Preprocessing Matters

Real-world text is **messy**:

- Mixed casing: `"The CAT sat"` vs `"the cat sat"`
- HTML artifacts: `"<b>Hello</b> &amp; welcome"`
- URLs, emails, phone numbers mixed with content
- Emojis, special characters, Unicode variants
- Contractions, abbreviations, slang

**Without preprocessing**, models see `"Hello"`, `"hello"`, and `"HELLO"` as **three different tokens**, inflating vocabulary size and fragmenting learned representations.

> **Note:** This notebook covers *pre-text processing* (cleaning and normalization **before** feeding text to a model). The companion notebook `06_Postprocessing_Decoding_Detokenization_Filtering.ipynb` covers *post-text processing* (what happens **after** a model generates output).

In [2]:
# A deliberately messy text sample
messy_text = """
<p>Hey!!! Check out https://example.com for MORE info 😊😊😊
Contact us at support@example.com or call 555-1234.
I can't believe it's only $9.99!!! Won't you agree?
The café résumé naïve file is at /tmp/data_v2.0.csv
RT @user: This is 100% the BEST thing ever!!! #NLP #awesome</p>
"""
print("Raw text:")
print(messy_text)

Raw text:

<p>Hey!!! Check out https://example.com for MORE info 😊😊😊
Contact us at support@example.com or call 555-1234.
I can't believe it's only $9.99!!! Won't you agree?
The café résumé naïve file is at /tmp/data_v2.0.csv
RT @user: This is 100% the BEST thing ever!!! #NLP #awesome</p>



## 2. Text Normalization Steps

### 2.1 Lowercasing

**When to lowercase:**
- Bag-of-Words, TF-IDF, topic modeling
- Simple sentiment analysis
- Search/retrieval tasks

**When NOT to lowercase:**
- Named Entity Recognition (NER): `"Apple"` (company) vs `"apple"` (fruit)
- Machine translation: proper nouns
- Modern transformer models (BERT etc.) that have their own casing logic

In [3]:
def to_lowercase(text: str) -> str:
    """Convert text to lowercase."""
    return text.lower()

sample = "The QUICK Brown FOX Jumps Over Apple Inc."
print(f"Before: {sample}")
print(f"After:  {to_lowercase(sample)}")
print()
print("Notice: 'Apple Inc.' loses its proper-noun signal after lowercasing.")

Before: The QUICK Brown FOX Jumps Over Apple Inc.
After:  the quick brown fox jumps over apple inc.

Notice: 'Apple Inc.' loses its proper-noun signal after lowercasing.


### 2.2 Removing URLs, Emails, HTML Tags

In [4]:
def remove_urls(text: str) -> str:
    """Remove HTTP/HTTPS URLs."""
    return re.sub(r"https?://\S+|www\.\S+", "", text)

def remove_emails(text: str) -> str:
    """Remove email addresses."""
    return re.sub(r"\S+@\S+\.\S+", "", text)

def remove_html_tags(text: str) -> str:
    """Strip HTML tags and decode common entities."""
    text = re.sub(r"<[^>]+>", "", text)
    # Decode common HTML entities
    text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
    text = text.replace("&quot;", '"').replace("&#39;", "'")
    return text

sample = '<p>Visit https://example.com or email user@test.com &amp; enjoy!</p>'
print(f"Before:        {sample}")
print(f"No URLs:       {remove_urls(sample)}")
print(f"No emails:     {remove_emails(sample)}")
print(f"No HTML:       {remove_html_tags(sample)}")
# Chained
cleaned = remove_html_tags(remove_emails(remove_urls(sample)))
print(f"All combined:  {cleaned}")

Before:        <p>Visit https://example.com or email user@test.com &amp; enjoy!</p>
No URLs:       <p>Visit  or email user@test.com &amp; enjoy!</p>
No emails:     <p>Visit https://example.com or email  &amp; enjoy!</p>
No HTML:       Visit https://example.com or email user@test.com & enjoy!
All combined:  Visit  or email  & enjoy!


### 2.3 Handling Emojis

Two strategies:

| Strategy | When to use |
|----------|-------------|
| **Remove** | Topic modeling, information extraction |
| **Replace with text** | Sentiment analysis (emojis carry sentiment!) |

In [5]:
def remove_emojis(text: str) -> str:
    """Remove emoji characters."""
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002702-\U000027B0"  # dingbats
        "\U0000FE00-\U0000FE0F"  # variation selectors
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub("", text)

def replace_emojis_simple(text: str) -> str:
    """Replace common emojis with text descriptors."""
    emoji_map = {
        "\U0001F600": " happy ", "\U0001F60A": " happy ",
        "\U0001F622": " sad ", "\U0001F621": " angry ",
        "\U0001F44D": " thumbsup ", "\U0001F44E": " thumbsdown ",
        "\u2764": " love ", "\U0001F525": " fire ",
    }
    for emoji, replacement in emoji_map.items():
        text = text.replace(emoji, replacement)
    return text

sample = "Great product! \U0001F600\U0001F44D Love it \u2764"
print(f"Original:  {sample}")
print(f"Removed:   {remove_emojis(sample)}")
print(f"Replaced:  {replace_emojis_simple(sample)}")

Original:  Great product! 😀👍 Love it ❤
Removed:   Great product!  Love it 
Replaced:  Great product!  happy  thumbsup  Love it  love 


### 2.4 Number Handling

| Strategy | Use case |
|----------|----------|
| **Remove** | When numbers are noise (e.g., social media analysis) |
| **Replace with `<NUM>` token** | When the *presence* of a number matters but not its value |
| **Keep** | When numerical values are features (e.g., financial text) |

In [6]:
def remove_numbers(text: str) -> str:
    """Remove all digit sequences."""
    return re.sub(r"\d+", "", text)

def replace_numbers(text: str, token: str = "<NUM>") -> str:
    """Replace digit sequences with a placeholder token."""
    return re.sub(r"\d+", token, text)

sample = "Order #12345 shipped on 2024-01-15 for $29.99"
print(f"Original:  {sample}")
print(f"Removed:   {remove_numbers(sample)}")
print(f"Replaced:  {replace_numbers(sample)}")

Original:  Order #12345 shipped on 2024-01-15 for $29.99
Removed:   Order # shipped on -- for $.
Replaced:  Order #<NUM> shipped on <NUM>-<NUM>-<NUM> for $<NUM>.<NUM>


### 2.5 Expanding Contractions

In [7]:
CONTRACTIONS = {
    "won't": "will not",
    "can't": "cannot",
    "shouldn't": "should not",
    "wouldn't": "would not",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hasn't": "has not",
    "haven't": "have not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "i'm": "i am",
    "i've": "i have",
    "i'll": "i will",
    "i'd": "i would",
    "he's": "he is",
    "she's": "she is",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "we're": "we are",
    "they're": "they are",
    "we've": "we have",
    "they've": "they have",
    "we'll": "we will",
    "they'll": "they will",
    "let's": "let us",
    "what's": "what is",
    "who's": "who is",
}

def expand_contractions(text: str) -> str:
    """Expand English contractions."""
    # Work on lowercased version for matching
    for contraction, expansion in CONTRACTIONS.items():
        # Case-insensitive replacement
        text = re.sub(re.escape(contraction), expansion, text, flags=re.IGNORECASE)
    return text

sample = "I can't believe it's only $9.99! Won't you agree? They're amazing."
print(f"Before: {sample}")
print(f"After:  {expand_contractions(sample)}")

Before: I can't believe it's only $9.99! Won't you agree? They're amazing.
After:  I cannot believe it is only $9.99! will not you agree? they are amazing.


### 2.6 Unicode Normalization

Unicode allows multiple representations of the same character:
- `e\u0301` (e + combining acute) vs `\u00e9` (precomposed e-acute)

**NFC** (Canonical Decomposition + Canonical Composition) is the standard choice.

In [8]:
def normalize_unicode(text: str, form: str = "NFC") -> str:
    """Normalize Unicode text (NFC, NFD, NFKC, NFKD)."""
    return unicodedata.normalize(form, text)

def remove_accents(text: str) -> str:
    """Remove accent marks (useful for some search/retrieval tasks)."""
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

# Demonstrate: two ways to write 'e-acute'
s1 = "caf\u00e9"           # precomposed
s2 = "cafe\u0301"          # decomposed (e + combining accent)
print(f"s1: {s1!r}  len={len(s1)}")
print(f"s2: {s2!r}  len={len(s2)}")
print(f"Equal? {s1 == s2}")

# After NFC normalization
s1n = normalize_unicode(s1)
s2n = normalize_unicode(s2)
print(f"\nAfter NFC:")
print(f"s1: {s1n!r}  len={len(s1n)}")
print(f"s2: {s2n!r}  len={len(s2n)}")
print(f"Equal? {s1n == s2n}")

# Accent removal
print(f"\nAccent removal: {remove_accents('café résumé naïve')}")

s1: 'café'  len=4
s2: 'café'  len=5
Equal? False

After NFC:
s1: 'café'  len=4
s2: 'café'  len=4
Equal? True

Accent removal: cafe resume naive


## 3. Punctuation Handling

Punctuation handling is **context-dependent**:

| Context | Recommendation |
|---------|----------------|
| **BoW / TF-IDF** | Usually remove punctuation |
| **Sentiment analysis** | Keep `!`, `?` (they carry emphasis) |
| **Code/technical text** | Keep most punctuation |
| **Tokenizer input (BERT)** | Let the tokenizer handle it |

In [9]:
def remove_punctuation(text: str) -> str:
    """Remove all standard ASCII punctuation."""
    return text.translate(str.maketrans("", "", string.punctuation))

def remove_punctuation_selective(text: str, keep: str = "!?") -> str:
    """Remove punctuation except specified characters."""
    to_remove = "".join(c for c in string.punctuation if c not in keep)
    return text.translate(str.maketrans("", "", to_remove))

sample = "Wow!!! This is amazing... right?! Price: $9.99 (50% off)"
print(f"Original:               {sample}")
print(f"All punct removed:      {remove_punctuation(sample)}")
print(f"Keep ! and ?:           {remove_punctuation_selective(sample)}")

Original:               Wow!!! This is amazing... right?! Price: $9.99 (50% off)
All punct removed:      Wow This is amazing right Price 999 50 off
Keep ! and ?:           Wow!!! This is amazing right?! Price 999 50 off


## 4. Tokenization

**Tokenization** splits text into individual units (tokens). The choice of tokenizer significantly affects downstream model performance.

### 4.1 Whitespace Tokenization

The simplest approach: split on whitespace characters.

In [10]:
def whitespace_tokenize(text: str) -> List[str]:
    """Split text on whitespace."""
    return text.split()

sample = "The cat sat on the mat."
tokens = whitespace_tokenize(sample)
print(f"Input:  {sample!r}")
print(f"Tokens: {tokens}")
print(f"Count:  {len(tokens)}")
print()
print("Problem: 'mat.' is one token -- punctuation is attached!")

Input:  'The cat sat on the mat.'
Tokens: ['The', 'cat', 'sat', 'on', 'the', 'mat.']
Count:  6

Problem: 'mat.' is one token -- punctuation is attached!


### 4.2 Word Tokenization (Rule-Based)

Splits on whitespace **and** separates punctuation into its own tokens.

In [11]:
def word_tokenize(text: str) -> List[str]:
    """Rule-based word tokenizer: separates punctuation from words."""
    # Insert spaces around punctuation
    text = re.sub(r"([^\w\s])", r" \1 ", text)
    # Collapse whitespace and split
    return text.split()

sample = "The cat sat on the mat. It's $9.99!"
print(f"Whitespace: {whitespace_tokenize(sample)}")
print(f"Word-level: {word_tokenize(sample)}")
print()
print("Notice: punctuation is now separate tokens.")
print("Caveat: '$9.99' becomes ['$', '9', '.', '99'] -- may not be desired.")

Whitespace: ['The', 'cat', 'sat', 'on', 'the', 'mat.', "It's", '$9.99!']
Word-level: ['The', 'cat', 'sat', 'on', 'the', 'mat', '.', 'It', "'", 's', '$', '9', '.', '99', '!']

Notice: punctuation is now separate tokens.
Caveat: '$9.99' becomes ['$', '9', '.', '99'] -- may not be desired.


### 4.3 Sentence Tokenization

Splits text into sentences. A simple rule-based approach uses `.`, `!`, `?` as delimiters, but must handle abbreviations.

In [12]:
def sentence_tokenize(text: str) -> List[str]:
    """Simple rule-based sentence tokenizer."""
    # Split on sentence-ending punctuation followed by whitespace and uppercase
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [s.strip() for s in sentences if s.strip()]

text = (
    "Dr. Smith went to Washington. He arrived at 3 p.m. "
    "The meeting was productive! Will there be a follow-up? "
    "Yes, next Tuesday."
)
sentences = sentence_tokenize(text)
print("Sentences found:")
for i, s in enumerate(sentences, 1):
    print(f"  {i}. {s}")
print()
print("Note: 'Dr.' and 'p.m.' are tricky abbreviations for rule-based splitters.")

Sentences found:
  1. Dr.
  2. Smith went to Washington.
  3. He arrived at 3 p.m.
  4. The meeting was productive!
  5. Will there be a follow-up?
  6. Yes, next Tuesday.

Note: 'Dr.' and 'p.m.' are tricky abbreviations for rule-based splitters.


## 5. Using `src/utils/text_preprocessing.py`

Our repository provides pre-built utilities. Let us use them.

In [13]:
from src.utils.text_preprocessing import (
    normalize_text,
    basic_tokenize,
    build_vocab,
    texts_to_sequences,
)

# 1. Normalize
raw = "I can't believe https://example.com is down! Email admin@site.com ASAP!"
cleaned = normalize_text(
    raw,
    lowercase=True,
    remove_urls=True,
    remove_emails=True,
    expand_contractions=True,
)
print(f"Raw:     {raw}")
print(f"Cleaned: {cleaned}")
print()

# 2. Tokenize
tokens = basic_tokenize(cleaned)
print(f"Tokens: {tokens}")
print()

# 3. Build vocabulary from a small corpus
corpus = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "Cats and dogs are pets.",
]
vocab = build_vocab(corpus, min_freq=1)
print(f"Vocabulary ({len(vocab)} tokens): {vocab}")
print()

# 4. Convert to integer sequences
sequences = texts_to_sequences(corpus, vocab, max_len=8)
for text, seq in zip(corpus, sequences):
    print(f"  {text:40s} -> {seq}")

Raw:     I can't believe https://example.com is down! Email admin@site.com ASAP!
Cleaned: i cannot believe is down! email asap!

Tokens: ['i', 'cannot', 'believe', 'is', 'down', '!', 'email', 'asap', '!']

Vocabulary (15 tokens): {'<PAD>': 0, '<UNK>': 1, 'the': 2, '.': 3, 'sat': 4, 'on': 5, 'cat': 6, 'mat': 7, 'dog': 8, 'log': 9, 'cats': 10, 'and': 11, 'dogs': 12, 'are': 13, 'pets': 14}

  The cat sat on the mat.                  -> [2, 6, 4, 5, 2, 7, 3, 0]
  The dog sat on the log.                  -> [2, 8, 4, 5, 2, 9, 3, 0]
  Cats and dogs are pets.                  -> [10, 11, 12, 13, 14, 3, 0, 0]


## 6. Building a Preprocessing Pipeline

A **pipeline** chains preprocessing steps in a configurable, reproducible order.

In [14]:
class TextPreprocessingPipeline:
    """Configurable text preprocessing pipeline."""

    def __init__(self, steps: List[Callable[[str], str]]):
        """
        Args:
            steps: Ordered list of functions, each taking and returning a string.
        """
        self.steps = steps

    def __call__(self, text: str) -> str:
        for step in self.steps:
            text = step(text)
        return text

    def process_batch(self, texts: List[str]) -> List[str]:
        return [self(text) for text in texts]

    def show_steps(self, text: str) -> None:
        """Show the output after each step (useful for debugging)."""
        print(f"Input: {text!r}")
        print("-" * 60)
        for step in self.steps:
            text = step(text)
            print(f"{step.__name__:30s} -> {text!r}")
        print("-" * 60)
        print(f"Output: {text!r}")


# Build a pipeline for sentiment analysis
sentiment_pipeline = TextPreprocessingPipeline([
    remove_html_tags,
    remove_urls,
    remove_emails,
    expand_contractions,
    to_lowercase,
    remove_emojis,
    lambda text: replace_numbers(text, "<NUM>"),
    lambda text: re.sub(r"\s+", " ", text).strip(),  # collapse whitespace
])
# Give lambdas names for display
sentiment_pipeline.steps[-2].__name__ = "replace_numbers"
sentiment_pipeline.steps[-1].__name__ = "collapse_whitespace"

sentiment_pipeline.show_steps(messy_text)

Input: "\n<p>Hey!!! Check out https://example.com for MORE info 😊😊😊\nContact us at support@example.com or call 555-1234.\nI can't believe it's only $9.99!!! Won't you agree?\nThe café résumé naïve file is at /tmp/data_v2.0.csv\nRT @user: This is 100% the BEST thing ever!!! #NLP #awesome</p>\n"
------------------------------------------------------------
remove_html_tags               -> "\nHey!!! Check out https://example.com for MORE info 😊😊😊\nContact us at support@example.com or call 555-1234.\nI can't believe it's only $9.99!!! Won't you agree?\nThe café résumé naïve file is at /tmp/data_v2.0.csv\nRT @user: This is 100% the BEST thing ever!!! #NLP #awesome\n"
remove_urls                    -> "\nHey!!! Check out  for MORE info 😊😊😊\nContact us at support@example.com or call 555-1234.\nI can't believe it's only $9.99!!! Won't you agree?\nThe café résumé naïve file is at /tmp/data_v2.0.csv\nRT @user: This is 100% the BEST thing ever!!! #NLP #awesome\n"
remove_emails                  ->

In [15]:
# Process a batch of texts
batch = [
    "<b>Great</b> product! Won't return it. https://review.com",
    "Terrible service... I can't believe this! Email: angry@user.com",
    "It's ok, nothing special. 3/5 stars.",
]

processed = sentiment_pipeline.process_batch(batch)
print("Processed batch:")
for orig, proc in zip(batch, processed):
    print(f"  IN:  {orig}")
    print(f"  OUT: {proc}")
    print()

Processed batch:
  IN:  <b>Great</b> product! Won't return it. https://review.com
  OUT: great product! will not return it.

  IN:  Terrible service... I can't believe this! Email: angry@user.com
  OUT: terrible service... i cannot believe this! email:

  IN:  It's ok, nothing special. 3/5 stars.
  OUT: it is ok, nothing special. <NUM>/<NUM> stars.



## 7. Common Mistakes & Debugging Tips

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Lowercasing for NER | Loses proper-noun signals | Skip lowercasing for NER |
| Removing all punctuation for sentiment | Loses `!`, `?` emphasis | Use selective punctuation removal |
| Removing numbers in financial text | Loses critical data | Keep or replace with `<NUM>` |
| Not normalizing Unicode | Same word has multiple representations | Apply NFC normalization early |
| Removing stop words for deep learning | Breaks context ("not good" -> "good") | Skip stop word removal for contextual models |
| Applying preprocessing after subword tokenization | Corrupts subword tokens | Always preprocess **before** tokenization |
| Not collapsing whitespace | Multiple spaces create empty tokens | Always `re.sub(r"\s+", " ", text).strip()` as last step |

**Debugging tip:** Use `pipeline.show_steps(text)` to see the effect of each step.

## 8. Exercises

### Exercise 1: Preprocess a Messy Dataset

Given the following messy dataset, build an appropriate preprocessing pipeline and clean all texts.

In [16]:
messy_dataset = [
    "<div>OMG!!! 😊 Check https://sale.com for 50% OFF!! #shopping</div>",
    "I WON'T buy from them again. Emailed complaints@store.com 3 times!!!",
    "The résumé was saved in /docs/résumé_v2.pdf ... naïve approach didn't work",
    "RT @user123: This café has the BEST latte!!! \u2764\u2764\u2764 $4.50 only",
    "  <p>Multiple   spaces   and\ttabs\nand newlines  </p>  ",
]

# TODO: Build a pipeline and process this dataset
# Your code here

### Exercise 2: Custom Tokenizer

Write a tokenizer that:
1. Keeps contractions as single tokens (`"don't"` stays `"don't"`)
2. Keeps currency amounts together (`"$9.99"` stays `"$9.99"`)
3. Separates other punctuation

In [17]:
# TODO: Implement custom tokenizer
# def custom_tokenize(text: str) -> List[str]:
#     ...

# Test cases:
# custom_tokenize("I don't have $9.99 for this!") 
# Expected: ["I", "don't", "have", "$9.99", "for", "this", "!"]

## 9. Exercise Solutions

In [18]:
# ---- Exercise 1 Solution ----

exercise_pipeline = TextPreprocessingPipeline([
    remove_html_tags,
    remove_urls,
    remove_emails,
    normalize_unicode,           # Fix unicode variants
    expand_contractions,
    to_lowercase,
    remove_emojis,
    lambda text: replace_numbers(text, "<NUM>"),
    lambda text: re.sub(r"#\w+", "", text),          # Remove hashtags
    lambda text: re.sub(r"@\w+", "", text),          # Remove @mentions
    lambda text: re.sub(r"RT\s*", "", text, flags=re.IGNORECASE),  # Remove RT
    lambda text: re.sub(r"\s+", " ", text).strip(),  # Collapse whitespace
])

# Name lambdas for clarity
exercise_pipeline.steps[7].__name__ = "replace_numbers"
exercise_pipeline.steps[8].__name__ = "remove_hashtags"
exercise_pipeline.steps[9].__name__ = "remove_mentions"
exercise_pipeline.steps[10].__name__ = "remove_RT"
exercise_pipeline.steps[11].__name__ = "collapse_whitespace"

print("=" * 60)
print("Exercise 1: Messy Dataset Preprocessing")
print("=" * 60)
for i, text in enumerate(messy_dataset, 1):
    cleaned = exercise_pipeline(text)
    print(f"\n[{i}] Original: {text}")
    print(f"    Cleaned:  {cleaned}")

Exercise 1: Messy Dataset Preprocessing

[1] Original: <div>OMG!!! 😊 Check https://sale.com for 50% OFF!! #shopping</div>
    Cleaned:  omg!!! check for <NUM>% off!!

[2] Original: I WON'T buy from them again. Emailed complaints@store.com 3 times!!!
    Cleaned:  i will not buy from them again. emailed <NUM> times!!!

[3] Original: The résumé was saved in /docs/résumé_v2.pdf ... naïve approach didn't work
    Cleaned:  the résumé was saved in /docs/résumé_v<NUM>.pdf ... naïve approach did not work

[4] Original: RT @user123: This café has the BEST latte!!! ❤❤❤ $4.50 only
    Cleaned:  <NUM>: this café has the best latte!!! $<NUM>.<NUM> only

[5] Original:   <p>Multiple   spaces   and	tabs
and newlines  </p>  
    Cleaned:  multiple spaces and tabs and newlines


In [19]:
# ---- Exercise 2 Solution ----

def custom_tokenize(text: str) -> List[str]:
    """Tokenizer that preserves contractions and currency amounts."""
    # Pattern groups (order matters):
    # 1. Currency amounts: $12.34
    # 2. Contractions: don't, I'm, they're
    # 3. Words: sequences of word characters
    # 4. Punctuation: individual non-space, non-word characters
    pattern = r"\$\d+(?:\.\d+)?|\w+'\w+|\w+|[^\s\w]"
    return re.findall(pattern, text)

print("\n" + "=" * 60)
print("Exercise 2: Custom Tokenizer")
print("=" * 60)
test_cases = [
    "I don't have $9.99 for this!",
    "She's buying a $100.50 gift, isn't she?",
    "They're offering 50% off -- amazing!",
]
for text in test_cases:
    tokens = custom_tokenize(text)
    print(f"  Input:  {text}")
    print(f"  Tokens: {tokens}")
    print()


Exercise 2: Custom Tokenizer
  Input:  I don't have $9.99 for this!
  Tokens: ['I', "don't", 'have', '$9.99', 'for', 'this', '!']

  Input:  She's buying a $100.50 gift, isn't she?
  Tokens: ["She's", 'buying', 'a', '$100.50', 'gift', ',', "isn't", 'she', '?']

  Input:  They're offering 50% off -- amazing!
  Tokens: ["They're", 'offering', '50', '%', 'off', '-', '-', 'amazing', '!']

